In [3]:
import torch
from torch import nn
from torch.nn import functional as F
from tokenizers import Tokenizer
from datasets import load_dataset

In [4]:
torch.manual_seed(2026)

device = "cuda" if torch.cuda.is_available() else "cpu"

batch_size = 64
block_size = 256 # max context length
n_embd = 384
lr = 3e-4
eval_iters = 200
eval_interval = 500
max_iters = 5000
dropout = 0.2
n_dlayers = 6
n_elayers = 6
n_heads = 6
vocab_size = 16384
n_blocks = 6

In [5]:
tokenizer = Tokenizer.from_file("tokenizer-wiki.json")

ds = load_dataset("Salesforce/wikitext", "wikitext-103-raw-v1")

In [10]:
dtr = ds["train"]["text"]

In [99]:
def get_batch(split=None):
    data = dtr
    ix = torch.randint(len(data), (batch_size,))

    x_list = []
    y_list = []

    for i in ix:
        # Retrieve raw text segment and encode
        # We combine text chunk to ensure we have enough tokens, or slice carefully
        raw_text_x = " ".join(data[i : i + 10])
        tokens = tokenizer.encode(raw_text_x).ids

        # Ensure we have at least block_size + 1 tokens
        if len(tokens) < block_size + 1:
            # Pad with 0s if too short
            tokens = tokens + [0] * (block_size + 1 - len(tokens))

        x_chunk = tokens[:block_size]
        y_chunk = tokens[1:block_size + 1]

        x_list.append(torch.tensor(x_chunk, dtype=torch.long))
        y_list.append(torch.tensor(y_chunk, dtype=torch.long))

    x = torch.stack(x_list).to(device)
    y = torch.stack(y_list).to(device)

    return x, y

In [97]:
get_batch()

torch.Size([21071])


/tmp/ipykernel_932/1430789363.py:10: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x, y = torch.tensor(x, dtype=torch.long, device=device), torch.tensor(y, dtype=torch.long, device=device)


(tensor([6383, 5471, 5027,  ..., 6807,    1,   13], device='cuda:0'),
 tensor([6383, 5471, 5027,  ...,   28,   28,   28], device='cuda:0'))

In [109]:
class Transformer(nn.Module):
    def __init__(self):
        super().__init__()
        self.embd = nn.Embedding(vocab_size, n_embd)
        # Add learnable positional embeddings
        self.pos_embd = nn.Embedding(block_size, n_embd)

        # Initialize MultiheadAttention with batch_first=True
        self.mha1 = nn.MultiheadAttention(n_embd, n_heads, dropout, batch_first=True)
        self.mha2 = nn.MultiheadAttention(n_embd, n_heads, dropout, batch_first=True)

        self.ffwd = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)
        self.ln3 = nn.LayerNorm(n_embd)

        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        device_idx = idx.device

        # Token embeddings + Positional embeddings
        tok_emb = self.embd(idx.long())
        pos_emb = self.pos_embd(torch.arange(T, device=device_idx))
        x = tok_emb + pos_emb

        # mha1 takes (batch, seq_len, emb_dim)
        x, weights = self.mha1(x, x, x)
        x = self.ln1(x)

        # mha2 takes (batch, seq_len, emb_dim)
        x, weights = self.mha2(x, x, x)
        x = self.ln2(x)

        x = self.ffwd(x)
        x = self.ln3(x)

        logits = self.lm_head(x)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B * T, C)
            targets = targets.view(-1)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]
            logits, loss = self(idx_cond)
            logits = logits[:, -1, :]

            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)

        return idx

In [2]:
class GPT(nn.Module):
    def __init__(self, n_blocks):
        super().__init__()

        self.blocks = nn.Sequential(*[Transformer() for _ in range(n_blocks)])

    def forward(self, idx, targets=None):
        logits = self.blocks(idx)

        if targets is None:
            loss = None

        else:
            B, T, C = logits.shape
            logits = logits.view(B * T, C)
            targets = targets.view(-1)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]
            logits, loss = self(idx_cond)
            logits = logits[:, -1, :]

            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)

        return idx

NameError: name 'nn' is not defined

In [110]:
model = Transformer().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

In [114]:
context = tokenizer.encode("Hello world").ids
# logits, loss = model(torch.tensor(context, dtype=torch.long, device=device))

output = model.generate(torch.tensor(context, dtype=torch.long, device=device).unsqueeze(0), max_new_tokens=100)
tokenizer.decode(output[0].tolist())

'Hel lo world about from Hall islands ’ red obtained " and shield out people s , in armoured lot squ Wor ats n Pr 1 play football . June ge thir Storm in further Ar might = ) times brought Film material ( formed company the A for sp . According cord profile , lemur was her They on mm ly being in N od who @-@ more , any . = s , that . ceased tight has " and of as Moore ploy tr ) jo 100 od as to 1 . ker to silver \' , ony . Kate'

In [43]:
get_batch()

TypeError: expected Tensor as element 0 in argument 0, but got list

In [113]:
for step in range(max_iters):
    Xb, yb = get_batch()

    logits, loss = model(Xb, yb)

    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

    if step % 10 == 0:
        print(f'{step} / {max_iters}: {loss.item()}')

print("Final loss:", loss)

0 / 5000: 9.874394416809082
10 / 5000: 6.99359130859375
20 / 5000: 7.188369274139404
30 / 5000: 7.15364933013916
40 / 5000: 7.250436782836914
50 / 5000: 7.036638259887695
60 / 5000: 6.829656600952148
70 / 5000: 6.843786239624023
80 / 5000: 6.917026519775391


KeyboardInterrupt: 